# Mechanical Engineering Calculations for non-Mechanical Engineers

Victor J. Marrero/567
updated: 04/03/2024

Notebook based on MechanicalEngCalculationScriptRevA.mlx

Revision B: Switch functions Data arrangement from Array to Dictionaries

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import UnitsConversions as units
import MechanicalEngModule as Mech

## CNC Design Guidelines

### Inside Corner Fillets

When it comes to inside corner fillets, the radii size is key. Sizing corner radii appropriately can improve not only cutting efficiency and cost but quality as well. Consider the following for internal corner radii in your designs:

* Use a radius that does not correlate with standard tool sizes
* Radii should be above 1/32”
* Use as large a radius as possible
* Avoid small, deep radii

Most cutting tools come in standard fractional sizes, such as ¼”, ⅛”, etc. **Avoid these standard sizes; if the tool radii match the designed corner radii, it will not have the proper clearance to turn into the cut. Instead, the tool must come to a complete stop, pivot 90 degrees, then resume cutting. These abrupt cutting paths reduce efficiency and lead to quality issues such as chatter.**

<img src="images/CNCFigure1.JPG" height = 300>

Pro Tip:
Use a radius 1.3 times the radius of the closest standard tool size and aim for a radii-to-depth ratio of 1:4 for pocket radii.
Pro Tip:
For better manufacturability of floor fillets, use a standard bull nose mill radius. Should be less than wall radius.

#### Holes
Pro Tip:
Pick hole sizes that work universally. Lowering hole size variation means the same tool can be used for multiple features, reducing cycle time and cost.
Pro Tip:
Keep your drill hole diameter-to-depth ratio below 1:10 for manufacturability. Even better is a time and cost-saving ratio of 1:4. For example, a 0.250” diameter drilled hole at 1” deep is optimal, while a depth of 2.5” is achievable.

In [ ]:


ToolDiameter = np.array([1/32, 3/64, 1/16,5/64,3/32,7/64,1/8,9/64,5/32,11/64,3/16,13/64,7/32,15/64,1/4,17/64,9/32,19/64,1/2])
D_tool = ToolDiameter[2]  # [in]

toolRadius = D_tool/2
edgeRadius = 1.3*toolRadius
edgeRadius_mm = units.in2mm(edgeRadius)
print('Edge radius based on tool diameter %2.3f [in]: %2.3f [in]' % (D_tool,edgeRadius))
depth = 4*edgeRadius
depth_mm = units.in2mm(depth)

print('max depth for given tool diameter: %2.3f [in]' % depth)

depth_known = .9
edgeRadius = depth_known/4
toolRadiusNeeded = edgeRadius/1.3



## CNC tool guidelines

Here are some recommended surface feet per minute (SFM) values for different materials:
 * Magnesium and its alloys: 250–400 SFM 
 * Monel metal or high-nickel steel, stainless steel: 30–50 SFM 
* Plastics or similar materials: 100–300 SFM 
* Mild .2 carbon to .3 carbon steel, .4 carbon to .5 carbon tool, 1.2 carbon forgings alloy - 300 to * 400 Brinell: 80–110, 70–80, 50–60, 40–50, 20–30 SFM 
* High carbon steel: 65–100 SFM 
* Free-machining steel: 100–150 SFM 
* Stainless steel, C1 302, 304: 60 SFM 
* Stainless steel, C1 310, 316: 70 SFM 
* Aluminum and its alloys: 250 SFM 
* Brass (360 high machining): 250 SFM 
* Bronze (high tensile): 100 SFM 
* Cast iron (soft): 100 SFM 

SFM is a unit of measurement that represents the linear distance a cutting tool travels in one minute. It's similar to miles per hour (mph) for a car.

In [ ]:
N = 2  # number of cutting edges
SFM = 250  # [surface-feet/minute] per table above
ToolDiameter = 1/8  # [in]
SpindleSpeed = (SFM*12)/(ToolDiameter * np.pi) # RPM 
print(SpindleSpeed)
ChipLoad = .007 # [in/tooth] chip load per tooth
FeedRate = N*ChipLoad*SpindleSpeed #  inches per minute (ipm) 
print(FeedRate)

## Position Tolerance Calculations

### General fixed fastener formula

The general fixed fastener formula is used for determining position tolerance values that are equally distributed between the threaded hole and the clearance hole.
The general fixed fastener formula is shown here. This formula solves for the total amount of tolerance for the position tolerances on both the threaded holes and the clearance holes.

The total tolerance "T" is applied to each position tolerance. This results in an equal distribution of the available tolerance.
The variable "F" is the MMC of the fastener. It is usually equal to the stated thread major diameter.
$$
T = \frac{H-F}{2}
$$

where 
T = total position tolerance for assembly
H = MMC of the clearance hole
F = MMC of the fastener

When calculating the position tolerance values in a fixed fastener assembly, one of the fixed fastener formulas is used. Since we want an equal distribution of the tolerances, we'll use the general fixed fastener formula.
Because the function of the clearance holes is to assemble, the MMC modifier is used in the tolerance in the position tolerance.
A screw thread is a self-centering feature for the threaded holes; therefore, the RFS condition is specified in the position tolerance.
Since we are going to use one of the fixed fastener formulas, the projected tolerance zone modifier is used in the position tolerance on the threaded holes.

<img src="images/PositionTolerance1.jpg" height = 200>

Now we'll use the general fixed fastener formula to calculate the position tolerance values for the cover and housing holes.
The MMC of the clearance holes (H) is 14.4 and the MMC of an M14 X 2 bolt is 14 (F).
Using the general fixed fastener formula, the tolerance value for the position tolerances for the clearance hole and tapped hole is 0.2.

<img src="images/PositionTolerance2.jpg" height = 350>

Use code below to evaluate tolerance for your case:

In [ ]:

Bolt = Mech.BoltSize("#2","Socket Cap Head")  # MMC of fastener; NAS1352
F = Bolt["Diameter"]

#F = .186 # % MMC of pin; NASM16555
H = .105; # 0.203 [.008]MMC of clearance hole
# F = 14;   # MMC of fastener
T = ((H-F)/2)
print("hole position tolerance = %2.3f mm [%2.3f in]" %(units.in2mm(T),T))

### Bolt pattern tolerance visualization

NOTE: MMC of a hole is the minimum diameter.  MMC of a rod is the maximum diameter.

In [ ]:

Pos_tol = .008  # [in]
Feature_D = H # feature dimeter [in]

MMC =  H # H maximum mateial condition of feature [in];
s = -1  # -1 for hole; +1 for outside diameter
VC = MMC + s*Pos_tol
x0 = 0.0035*0  # [in]
y0 = 0.0007*0

r1 = MMC/2
r2 = VC/2
r3 = F/2

theta = np.linspace(0,360,361)

X1 = r1*np.cos(units.d2r(theta))
Y1 = r1*np.sin(units.d2r(theta))
X2 = r2*np.cos(units.d2r(theta))
Y2 = r2*np.sin(units.d2r(theta))
X3 = r3*np.cos(units.d2r(theta))
Y3 = r3*np.sin(units.d2r(theta))


plt.plot(X1+x0,Y1+y0)

if r2>r1:
    plt.plot(X2+x0,Y2+y0,'r')
else:
    plt.plot(X2+x0,Y2+y0)

plt.plot(X3+x0,Y3+y0,'k')
plt.plot(x0,y0,'k+')
# plot(x2,y2,'+')
# plot(x3,y3,'+')
plt.grid()
plt.legend(['MMC Hole size','Virtual Condition','MMC fastener'],loc = "upper left")
plt.xlabel('[inches]')
plt.ylabel('[inches]')
plt.title("Virtual condition hole fit check")
# square plot
plt.axis('square')
plt.show()


x0 = 0.00  # [in]
y0 = T  # [in]

Max1 = np.max((X1))
Max2 = np.max((X3+x0))

if Max2 > Max1:
    plt.plot(X1,Y1,'r')
    plt.plot(X3+x0,Y3+y0,'k')
    plt.plot(0,y0,'r+')
else:
    plt.plot(X1,Y1,color = 'tab:blue')
    plt.plot(X3+x0,Y3+y0,'k')
    plt.plot(0,y0,'+',color = 'tab:blue')

plt.plot(x0,y0,'k+')

# plot(x2,y2,'+')
# plot(x3,y3,'+')
plt.grid()
plt.legend(['MMC Hole size','MMC fastener'],loc = "upper left")
plt.xlabel('[inches]')
plt.ylabel('[inches]')
plt.title("Individual feature alingment test")
# square plot
plt.axis('square')
plt.show()

### Surface tolerance visualization


In [ ]:
### user input area for surface visualization  ####

SurfaceXPoints = np.array([8,12.5,14.254,18])
SurfaceYPoints = np.array([0,0,-4.813,-4.813])

dS = .127 # [mm] Surface tolerance
dx = dS/np.cos(units.d2r(20))
dy = dS/np.sin(units.d2r(20))

SurfaceToleranceXPointsOutSide = np.array([12.5+dx/2,14.254+dx/2])
SurfaceToleranceYPointsOutSide = np.array([0+dy/2,-4.813+dy/2])              
SurfaceToleranceXPointsInSide = np.array([12.5-dx/2,14.254-dx/2])
SurfaceToleranceYPointsInSide = np.array([0-dy/2,-4.813-dy/2])

plt.plot(SurfaceXPoints,SurfaceYPoints)
plt.plot(SurfaceToleranceXPointsOutSide,SurfaceToleranceYPointsOutSide,'red')
plt.plot(SurfaceToleranceXPointsInSide,SurfaceToleranceYPointsInSide,'red')
plt.grid()
plt.axis('equal')
plt.xlabel("[mm]")
plt.ylabel("[mm]")
plt.show()


#### Slot/pin tolerance visualization

This is used for high precision alignment.  Main datum reference shall be defined from the pin/sloth features and any position tolerance feature that depends form this alignment shall use these datums in the tolerance frame.

Remember that:
1) Virtual Conditions (VC) for a inner boundary is the smallest size (MMC) - Geometrical tolerance.
2) Virtual Conditions (VC) for a outer boundary is the largest size (MMC)  + Geometrical tolerance.

Note that the Geometrical tolerance includes the tolerance of both features to be evaluated. 



In [18]:

DowelPinDiameterNominal = .250  # average between max and min
DowelPinDiameterTolerance = .0001

# tolerance for slip-fit mate part (pin hole/slot)
DowelPinMateHoleDiameterMMC = DowelPinDiameterNominal + .001 
DowelPinMateHoleDiameterLMC = DowelPinMateHoleDiameterMMC + .0015 

## making the perpendicularity = 0 means that the limit for Virtual Condition == MMC limit;
#  in other words, if the hole its at its absolute smallest it cannot have a perpendicularity deviation.
DowelPinMateHoleDiameterPerpendicularityMMC = .0000

DowelPinMateHoleDiameterVC = DowelPinMateHoleDiameterMMC + DowelPinMateHoleDiameterPerpendicularityMMC



## Stack analysis
based on : Calculation of Tolerance Stacks Using Direct-Position Approach in Geometric Dimensioning and Tolerancing by Cheng Y. Lin and Nathan J. Luetke

The equations here are for the Regardles of Feature of Size (RFS) and Maximum Material Condition (MMC) case.

In [ ]:

Dim1 = 10 # MMC hole diameter
Dim2 = 20 # MMC hole diameter

Tol1 =  np.array([-0.254,0.254]) # Hole tolerance
Tol2 =  np.array([-0.127,0.127])

PosTol1 = .152
PosTol2 = .254

# MMC1 = Dim1-Tol1;
# MMC2 = Dim2 - Tol2;

L = 20  # basic dimension; distance between Dim1 and Dim2

#MMC calculated values
X_max_mmc = L - PosTol1/2 - PosTol2/2 - (Dim1+Tol1[1])/2 - (Dim2 + Tol2[1])/2
X_min_mmc = L - (PosTol1/2 + np.sum(np.abs(Tol1)))/2 - (PosTol2/2 + np.sum(np.abs(Tol2)))/2 - (Dim1+Tol1[1])/2 - (Dim2 + Tol2[1])/2

print('minimum calculated MMC value: %3.3f;  maximum calculated MMC value: %3.3f' % (X_min_mmc, X_max_mmc))
#RFS calculated values
X_max_rfs = L + PosTol1/2 + PosTol2/2 - (Dim1+Tol1[0])/2 - (Dim2 + Tol2[0])/2
X_min_rfs = L - PosTol1/2 - PosTol2/2 - (Dim1+Tol1[1])/2 - (Dim2 + Tol2[1])/2
print('minimum calculated MMC value: %3.3f;  maximum calculated MMC value: %3.3f' % (X_min_rfs, X_max_rfs))


### Arithmetic or Worst case Tolerance Staking

from Tolerance Stack Analysis Methods by Fritz Scholz 1995

The worst case  or arithmetic method is the most conservative and will ensure 100% chance of assembly.  If the arithmetic stacking method gives ataisfactory assembly tolerance results, then there is little motivstion to try ny of the other methods, ecept to relax detail tilerances to acheive cost reduction.

Assuming $|\epsilon_1| = |L_i = \lambda_i| \leq T_i $ for all $i = 1,2,...,n$ we can bound $|G-\gamma|$ by

$$
T_{assy}^{arith} = \sum_{1}^{n} |a_i| T_i
$$

if $|a_i| = 1$ for all *i*, this simplifies to
$$
T_{assy}^{arith} = \sum_{1}^{n} T_i
$$

No mattter how the detail dimension $L_i$ deviate from their nominal values $\lambda_i$ within the contraint $|L_i - \lambda_i| \leq T_i$, the difference $|G-\gamma|$ is garantied to be bounded by $T_{assy}^{arith}$.

#### What is Tolerance Stack up?

Notes from (https://tolerancestackup.com/tolerance-stack-up/)

Tolerance Stack up is the permutation of inadequacy (or, flaws) of an assembly part design which directly affects the manufacturability of a part. Knowing the complications involved in assembling a part, tolerance stackup analysis anticipates the impact on total tolerance of a part based on assembly challenges and (preliminary) fluctuating tolerance of component.
Is the perfect fit, perfect enough? Tolerance Stackup Analysis answers this integral manufacturing question and anticipates design execution demand much in advance. Widely practiced across mechanical domain, tolerance stackup analysis quantifies the impact of gathered variations qualified by requirements (tolerances and dimensions) designated by assembly.

Often defined in engineering drawings, tolerances and dimensions of an assembly part often requires precise tolerance stackups. **Worst Case** and **Statistical Analysis (Root Sum Square/Root Mean Square)** are widely used methodologies for Tolerance Stackup Analysis which resorts to validation in single or double direction.  While the former calculates the maximal and minimal distance between two parts or attributes (based on worst cases), the latter intersects arithmetic and statistical methodologies to achieve similar objective.

#### Tolerance Stack up Methodology
To perform tolerance stackup analysis, there are two methods which are very common in the industry:
-Worst Case
-Root Sum Square (RSS)
-**Worst Case Method**

Worst Case method calculates the impact of shift in single (or, multiple) tolerance on the entire assembly. Popular for its simplistic nature, this method is based on the assumption that every dimension in the assembly chain would be manufactured at its maximum and minimum allowable value. Further, it is assumed that each deviation has adverse combination irrespective of its uncertainty. It is as simple as adding up the tolerances of the entire assembly chain, which is termed as linear sum.
Following are commonly used signs in worst case method:
$$
Accumulated \ Tolerance = (\Delta Y)
$$
n = Number of constituent dimensions in the dimension chain
$d_i$ = Tolerance associated with the ith dimension.

While it gives user the ease of calculating tolerance stackup analysis, this approach is only applicable when:
(a) The volume of production is very small
(b) 100 per cent acceptance is required
(c) The number of constituent dimensions in assembly is very small

##### Calculation
A worst-case tolerance analysis method is simple arithmetic (that’s right… just addition and subtraction), so let’s start there.
Say, we have an assembly of four thick plates like below:

<img src="images/Stackup.jpg" height = 300>

The above diagram depicts the tolerance and thickness of quadruple plates. Herein, the dimension and tolerance value needs to be found. In order to achieve this, the following steps need to be followed:

Calculate the lower specification limit (LSL) size for each of the plates like below:

For Plate 1:
LSL= 27-0.4 = 26.6

For Plate 2:
LSL= 15-0.3 = 14.7

For Plate 3:
LSL= 15-0.3 = 14.7

For Plate 4:
LSL= 15-0.5 = 14.5

On summing up the LSL thickness values of all plates, one would get the LSL thickness of the entire assembly as shown below:
$$
TL = 26.6 +14.7 + 14.7 + 14.5 = 70.5
$$
(TL = Total LSL thickness values)

Calculate the upper specification limit (USL) size for each of plate in the following manner:
For Plate 1:
USL= 27+0.4 = 27.4

For Plate 2:
USL= 15+0.3 = 15.3

For Plate 3:
USL= 15+0.3 = 15.3

For Plate 4:
USL= 15+0.5 = 15.5

On summing up the USL thickness values of all plates, one would get the USL of the entire assembly as show below:
$$
TU = 27.4 + 15.3 + 15.3 + 15.5 = 73.5
$$
(TU = Total USL thickness values)

Tolerance of the entire assembly obtained: 
$$
(TU – TL) / 2 =  (73.5-70.5)/2 =  1.5
$$

On summing up the nominal thickness dimensions of all plates, one would get the nominal thickness value of the entire assembly, as shown below:
$$
TN = 27 + 15 + 15 + 15 = 72
$$
(TN = Total Nominal)

So by worst case method one gets the overall dimension (X) of the assembly as:
$$
X = 72 ± 1.5
$$
Assembly tolerance stack-up analysis is used for calculating the tolerance value of the overall assembly (or a gap in the assembly) from the tolerance values of the individual components. The worst case method of the stack up analysis is the simplest.

In [ ]:
InputUnits = 'mm'
Tol = [.13] #[InputUnits]
NominalDim = np.array([47/2,40/2,33/2,25/2]) #[InputUnits]
UpperTolerance = np.array([.102,.102,.102,.102]) #[InputUnits]
LowerTolerance = np.array([.102,.102,.102,.102]) #[InputUnits]


TU = 0
TL = 0

T_assy = np.sum(Tol)

TU = np.sum(NominalDim + UpperTolerance)
TL = np.sum(NominalDim - LowerTolerance)

TotalTolerance = (TU-TL)/2
TotalNominal = np.sum(NominalDim)

print("worst case method for overall stackup tolerance of the assembly is: %3.3f [%s] +/- %3.3f [%s]" % (TotalNominal,InputUnits,TotalTolerance,InputUnits))


## Thermal Calculations:  tread carefully here...

### Heat Transfer: Convection (Ambient)

In [ ]:

h = 5; # [W/m^2] (natural convection)

Q = 3; # [W] input power

#dB(Q*1000)

factor = 1e6;  # mm^2 to m^2 factor

To = 20 # [C]  linspace(-60,20,21);
T_ss = To + 13; # [C]
#print(T_ss)
dT = T_ss-To #  [C]  T_ss-To;
A = Q/h/(dT)
A_min_in = units.mm2in(units.mm2in(A*factor))  # [in]

A_min = A*factor  #[mm]

A_est = 15*15
print('Minimum absorber surface area maximum steady state Temperature of %2.2f C: %2.3f [in^2]'% (T_ss,A_min_in))

Abs_power_handling = 1/(25.4*2); # [W/mm^2]; = 1 W/in^2  = 0.2 W/cm^2
Pwr_handling_margin = 1.5;  # Safety margin 
A_tot = A_est #[mm];

dT = Q/h/(A_tot) # [C]

T_ss = To+dT # [C] steady state temperature of hat coupler
#print(T_ss)
Min_Abs_area = Q/Abs_power_handling*Pwr_handling_margin #[mm^2]
Min_Abs_area_in = units.mm2in(units.mm2in(Min_Abs_area)) # [in^2]

print('Minimum absorber surface area based on Absorber power handling is: %2.3f [in^2]' % (Min_Abs_area_in))

### Heat Transfer: Radiation (Vacuum)

Calc = 1 :: estimates needed surface area for given steady state temperature

Calc = 0 :: estimates steady state Temperature

In [ ]:
Pin_dBm = float(35)

Q = units.MagP(Pin_dBm)/1e3 # [W]

sigma = 5.6703e-8 # [W/(m^2 K^4)] stefan-Boltzmann 

Emissivity = 0.1  # 1 = ideal radiator; 0.85 for AL Anodized (red or black); 0.19 for AL Chem Film
To = units.C2K(float(20)) # [K]  

# Calc = 1 :: estimates needed surface area for given steady state temperature
# Calc = 0 :: estimates steady state Temperature

Calc = 1
T_ss = To + units.C2K(13)  # [K]; 
A_tot = float(225)/factor   # [m^2]

if (Calc==1):  # estimate needed surface area for given steady state temperature

    #T_ss = To + C2K(13); # [K]; 
    A = Q/(Emissivity*sigma*(T_ss**4 - To**4)) # [m^2]
    A_mm = A*factor
    A_in = units.mm2in(units.mm2in(A_mm))

    print('Estimated needed surface area tor a given steady state temperature is: %2.3f [mm] (%2.3f [in])' % (A_mm,A_in))

else: # estimate steady state temperature for a given surface area
   
    T_ss = np.sqrt(np.sqrt(Q/(A_tot*Emissivity*sigma) + To**4)) 

    # T_ss = To + dT
    units.K2C(T_ss)

    print('Estimated Steady state temperature is: %2.3f [C]' % (T_ss))


### Emmisivity Area estimate

For TVac tests, the ideal scenario is to have a test article with the same thermal properties as the Flight configuration. If the test is using the EDU, it is very likely that the test article will be bare aluminum.

If we need to mimic the flight configuration, we could use kapton tape to increase the aluminum emissivity. one can estimate the total area that needs to be covered to achieved the final total emissivity of the part.

In [ ]:
AUTsurfaceArea = 129 # units^2
emmisivityKapton = .95
emmisivityAluminum_bare = .88
emmisivityAluminum_anodized = .9
emmisivityAluminum_iridite = .19

TapeArea = AUTsurfaceArea*(emmisivityAluminum_bare-emmisivityKapton)/(emmisivityAluminum_iridite-emmisivityAluminum_bare) # units^2

print(TapeArea)


	
## Calculating Grms (Root-Mean-Square Acceleration)

 It is very easy to describe the $G_{rms}$ (root-mean-square acceleration, sometimes written as GRMS or Grms or grms or grms) value as just the square root of the area under the ASD (Acceleration Spectral Density) vs. frequency curve, which it is. But to physically interpret this value we need to look at $G_{rms}$ a different way. The easiest way to think of the $G_{rms}$ is to first look at the mean square acceleration.

 **Mean-square acceleration is the average of the square of the acceleration over time.** That is, if you were to look at a time history of an accelerometer trace and were to square this time history and then determine the average value for this squared acceleration over the length of the time history, that would be the mean square acceleration. Using the mean square value keeps everything positive.

 The $G_{rms}$ is the root-mean-square acceleration (or rms acceleration), which is just the square root of the mean square acceleration determined above.

 If the accelerometer time history is a pure sinusoid with zero mean value, e.g., a steady-state vibration, the rms acceleration would be .707 times the peak value of the sinusoidal acceleration (if just a plain average were used, then the average would be zero). *If the accelerometer time history is a stationary Gaussian random time history, the rms acceleration (also called the 1 sigma acceleration) would be related to the statistical properties of the acceleration time history (you may have to refresh your probability and statistics knowledge for this)*:

    - 68.3% of the time, the acceleration time history would have peaks that would not exceed the +/- 1 sigma accelerations.
    - 95.4% of the time, the acceleration time history would have peaks that would not exceed the +/- 2 sigma accelerations.
    - 99.7% of the time, the acceleration time history would have peaks that would not exceed the +/- 3 sigma accelerations.

There is no theoretical maximum value for the Gaussian random variable; however, we typically design to 3 sigma since it would only be theoretically exceeded 0.3% of the time. In addition, from a practical point of view, we know that it would be physically impossible to achieve unreasonably high sigma values.


Below is presented the method to calculating the root-mean-square acceleration (Grms) response from a random vibration ASD curve.

Typical random vibration response curve:

<img src="images/rand_ex.gif" height = 300>

Grms values are determined by the square root of the area under a ASD vs. frequency response curve. The Acceleration Spectral Density values are in $g^2/Hz$ and the frequencies are in Hz.

<img src="images/rand_ex3.gif" height = 300>

The figure above shows a bandwidth of 10 Hz, which will be used as an example for calculating Grms.

**First**, calculate the number of octaves. From the plot, FL = 20Hz and FH = 30Hz. The equations below gives #octaves = 0.58.

$$
\#Octaves = \frac{log(F_H / F_L)}{log(2)}
$$

**Second**, calculate the dB value. For FL = 20Hz, ASDL = 1.0 g2/Hz, while ASDH = 1.1 g2/Hz. The calculated value is 0.41 dB. 

$$
dB = 10log(ASD_H / ASD_L)
$$

**Third**, calculate the slope, m, of the segment between the frequencies FL and FH. Dividing the number of dB by the number of octaves gives m = 0.71 dB/oct.

$$
m = dB/\#octaves
$$

**Fourth**, calculate the area under the curve between the frequencies FL and FH. In our example, $A = 10.53 g^2$

$$
A = 10log(2)\frac{ASD_H}{10log(2) + m}\left[F_H - F_L\left(\frac{F_L}{F_H}\right)^ {(m/10log(2))}\right]
$$

NOTE - The above equation is invalid if the slope m = -10log(2) because you would be dividing by zero.
**If m = -10log(2), use the following equation for the area**:

$$
A = ASD_L \times F_L \times ln(F_H/F_L)
$$

**Finally**, take the square root of the area for the Grms value. To finish our example, the acceleration = 3.24 Grms.

$$
G_{rms} = \sqrt(A) 
$$

In order to calculate the Grms value for the entire curve, sum up all the areas (A1 + A2 + A3 + ... + An = A) and take the square root of the sum.

**NOTE**: 3dB is a factor of 2 for ASD curves (g2/Hz) while 6dB is a factor of 2 for Grms values. For example, reducing a peak ASD value of 12g2/Hz by -3dB would give you 6g2/Hz; reducing a value of 12Grms -3dB results in a value of 9Grms and reducing it -6dB results in a value of 6Grms. *This tends to be confusing for people new to random vibration.*



In [ ]:

Frequency = np.array([20,50,700,800,925,2000]) # [Hz]

SafetyFactor = 1.9

# Random Vibe
ASD = np.array([0.01,0.015,0.015,0.03,0.03,0.00644])  # [G^2/Hz]

ASD_eval = ASD*SafetyFactor

Nf = Frequency.size
dB = np.zeros((Nf,), dtype=float, order='F')
Octave = np.zeros((Nf,), dtype=float, order='F')
Slope = np.zeros((Nf,), dtype=float, order='F')  # dB/Octave
Area = np.zeros((Nf,), dtype=float, order='F')
Grms = np.zeros((Nf,), dtype=float, order='F') # Acceleration

factor = 10*np.log10(2)

for i in range(1,Nf,1):
    Octave[i] = np.log10(Frequency[i]/Frequency[i-1]) / np.log10(2)
    dB[i] = 10*np.log10(ASD_eval[i]/ASD_eval[i-1])
    Slope[i] = dB[i]/Octave[i]
    if Slope[i] < 0:
        Area[i] = Area[i-1] + ASD_eval[i-1]*Frequency[i-1]*np.log(Frequency[i]/Frequency[i-1])
    else:
        Area[i] = Area[i-1] + 10*np.log10(2)*ASD_eval[i]/(10*np.log10(2)+Slope[i]) * (Frequency[i]-Frequency[i-1]*(Frequency[i-1]/Frequency[i])**(Slope[i]/(10*np.log10(2))))

Grms = np.sqrt(Area)
print(ASD_eval)
print(dB)
print(Octave)
print(Area)
print(Grms)

## Guideline for EMI Gasket Calculations

Strip / Flat Gaskets Fastener Requirements

### Applied Force
Most applications do not require more than 100 psi (0.69 MPa) to achieve an effective EMI seal. Waveguide flanges often provide ten times this amount. Hollow strips require less than 10 pounds per inch. 
Compression deflection data for many shapes, sizes and materials is included in the Performance Data section of this Design Guide.
The force required at the point of least pressure, generally midway between fasteners, can be obtained by using a large number of small fasteners spaced closely together. Alternatively, fasteners can be spaced further apart by using stiffer flanges and larger diameter bolts. Sheet metal parts require more fasteners per unit length than castings because they lack stiffness. 
To calculate average applied force required, refer to load-deflection curves for specific gasket materials and cross sections (see Performance Data).

### Fastener Size and spacing
Fastener spacing should be determined first. As a general rule, fasteners should not be spaced more than 2.0 inches (50 mm) apart for stiff flanges, and 0.75 inch (19 mm) apart for sheet metal if high levels of shielding are required. 
An exception to the rule is the spacing between fasteners found in large cabinet doors, which may vary from 3 inches (76.02 mm) between centers to single fasteners (i.e., door latches). The larger spacings are  compensated for by stiffer flange sections, very large gaskets, and/or some reduction in electrical performance requirements.  
The force per bolt is determined by dividing the total closure force by the number of bolts. Select a fastener with a stress value safely below the allowable stress of the fastener.

### Flange Deflection
The flange deflection between fasteners is a complex problem involving the geometry of the flange and the asymmetrical application of forces in
two directions. The one-dimensional solution, which treats the flange as a simple beam on an elastic foundation, is much easier to analyze and gives a
good first order approximation of the spacings required between fasteners, because most EMI gaskets are sandwiched between compliant flanges.

Variation in applied forces between fasteners can be limited to $\pm 10$ percent by adjusting the constants of the flange such that

$$
\beta = \left(\frac{K}{4 E_f I_f}\right)^{0.25}
$$

$K$ = foundation modulus of the seal 

$E_f$ = the modulus of elasticity of the flange

$I_f$ = the moment of inertia of the flange and seal

$d$ = spacing between fasteners

The moment of inertia ($I_f$ ) of rectangular sections, for example, may be obtained
from the following expression:

$$
I_f = \frac{b \times h^3 }{12}
$$

$b$ = the width of the flange in contact with the gasket (inches)
$h$ = the thickness of the flange (inches)

### Electrical Performance as a Function of Fastener Spacing

For bolt spacings equal to or approaching one-half wavelength at the highest operating frequency being considered, the shielding effectiveness at the point of least pressure is the governing value.
In determining the performance to be expected from a junction, find the value for an applied pressure which is 10% less (for $\beta d = 2$) than the value exerted by the bolts directly adjacent to the gasket.

the space between the fasteners can be considered to be a slot antenna loaded with a lossy dielectric. If the slot is completely filled, then the applied pressure must be 20% higher as cited. Conversely, if the slot is not completely filled (as shown in
Figure 1-16), the open area will be free to radiate energy through the slot.

<img src="images/Figure1-16.jpg" width = 600>

The cut-off frequency for polarizations parallel to the long dimension of the slot will be determined by the gap height, h. The cut-off frequency for the polarization vector perpendicular to the slot will be determined by the width of the slot, w. The attenuation through the slot is determined by the approximate formula:

$$
A[dB] = 54.5\frac{d}{\lambda_c}
$$

where 

d = the depth of the slot, and $\lambda_c$ is equal to 2w or 2h, depending upon the polarization being considered.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import UnitsConversions as units
import MechanicalToolbox as Mech

In [ ]:
#####  User input  ######
BoltNumber = "#4"
BoltType = 'Socket Cap Head' #  ; 'Flat Head 100 deg'
Bolt = Mech.BoltSize(BoltNumber,BoltType)  # MMC of fastener; NAS1352
Insert = Mech.InsertSize(BoltNumber)

Bolt_material_spec = Mech.MaterialDefinition("A286")
Gasket_material_spec = Mech.MaterialDefinition("Cho-seal 1285")
Parent_material_spec = Mech.MaterialDefinition("AL6061")
# gasket variables
GasketModulus = Gasket_material_spec["Young modulus"];  # [psi] assumed average modulus for seal per Cho-seal Document (Example Page9)
GasketThickness = .062;  # [in]
GasketWidth = units.mm2in(6);  # [in]

K = Mech.NutFactor['Lubricated']
Fastener_Engagement = 1.5;  # fastener engagement in diameters
if Bolt["Diameter"] < .112:
    FactorBolt = .6
else:
    FactorBolt = .65
FactorInsert = .65  # .75 for metals, .5 for polymers
FactorParentMaterial = .65  # .75 for metals, .5 for polymers

Torque = 0 # [lbs-in]

DesiredPressure = 200 # [psi]
PressureArea = 0.93  # [in^2]

######

D_maj = Bolt["Diameter"]
n = Bolt["Pitch"]
ys = Bolt_material_spec["Yield strength"] # yield strength of fastener;

if Torque == 0:
    A = np.round(np.pi/4*(D_maj-.9743/n)**2,3)  # rounding done to match PG table values; tensile stress area [in^2]

    P = np.round(FactorBolt*ys*A,1) # " 0.65 is 65% of yield strength of bolt material; Clamp force [lbf]
    print ('Bolt clamp force (pre-load) %3.3f [lbf]' % P)

    Torque = np.round(K*D_maj*P,1) # [lbf-in]
    print('Torque required for estimated pre-load: %3.3f [lbf-in]' % Torque)

Preload = round(Torque/(K*D_maj),1)

FlangeModulus  =  Parent_material_spec["Young modulus"] # Flange Modulus if elasticity
FlangeContactWidth = np.linspace(.01,.75,101)  # [in]
FlangeThickness = np.array([1/32,.04,1/16,3/32,1/4]) # [in] 

I_f = np.zeros([FlangeContactWidth.size,FlangeThickness.size])
Lgd_in = []
Lgd_mm = []

for i in range(FlangeThickness.size):
    I_f[:,i] = (FlangeContactWidth*FlangeThickness[i]**3)/12

    Lgd_in.append(('flange thickness = %6.4f [in]'% FlangeThickness[i]))
    Lgd_mm.append(('flange thickness = %6.4f [mm]'% units.in2mm(FlangeThickness[i])))

beta = (GasketModulus/(4*FlangeModulus*I_f))**0.25
beta_d = 2.25
BoltSeparation = beta_d/beta # [in]

plt.plot(FlangeContactWidth,BoltSeparation)
plt.grid()
plt.xlabel('Flange contact Width [in]')
plt.ylabel('Fastener spacing [in]')
plt.legend(Lgd_in,loc="upper left")
plt.show()

plt.plot(units.in2mm(FlangeContactWidth),units.in2mm(BoltSeparation))
plt.grid()
plt.xlabel('Flange contact Width [mm]')
plt.ylabel('Fastener spacing [mm]')
plt.legend(Lgd_mm,loc="upper left")
plt.show()


d_slot = units.in2m(.125)
w = units.in2m(.05)
h = units.in2m(.004)
lambda_c = np.array([2*w,2*h])

Attenuation_dB = 54.5*d_slot/lambda_c
print(Attenuation_dB)

MaxForcePerBolt = Preload
DesiredForce = DesiredPressure*PressureArea
N_bolts = np.ceil(DesiredForce/MaxForcePerBolt)

print("for the given EMI gasket area and bolt pre-load, a minimum of %2i bolts are required" % N_bolts)

## Contact Pressure and mounting torque

the information below is from CHO-THERM 1671 sped sheet.

The optimum contact pressure range for CHO-THERM materials is 300-500 psi (2.07 x 106 – 3.45 x 106N/m2).

Beyond this range,performance gains are negligible.

To convert mounting torque in contact pressure, use the following equation:

$
P =\frac{T \times N}{K \times D \times A}
$

P = Contact Pressure (psi or N/m2)\
T = Torque (in-lbs or N-m)\
N = Number of Fasteners\
K = Average Friction Factor = (0.2) \
D = Diameter (in. or m)\
A = Contact Area (in2 or m2)


In [ ]:

ys = 30e3 # [psi]; 30e3 for CRES 300; 120e3 for A286
BoltNumber = "3/8"
BoltType = 'Socket Cap Head' #  ; 'Flat Head 100 deg'
Bolt = Mech.BoltSize(BoltNumber,BoltType)  # MMC of fastener; NAS1352
n = 16  #pitch
K = Mech.NutFactor['Lubricated']
N = 6
D = Bolt["Diameter"]
A = np.pi*4**2 # in^2
A_bolt = .7854 *( D-.9743/n )**2
T = 70 # in-lbs
Force = T/K/D
MaxForce = np.round(.65*ys*A_bolt,1) # " 0.65 is 65% of yield strength of bolt material; Clamp force [lbf]
print ('Bolt clamp force (pre-load) %3.3f [lbf]' % Force)

MaxTorque = K*MaxForce*D 
print ('Max torque = %3.3f [in-lbs]' % MaxTorque)

P = N*Force/A
#P = T*N/(K*D*A)

print('Estimated applied pressure = %f [psi]'% P)


## Guideline for Bolted Joint Design and Analysis

from: Guideline for Bolted Joint Design and Analysis: Version 1.0, RoyMech.org and other sources

### Torque Estimates

from 540-PG-8072.1.2

The below equation assumes that:

- The joint fittings are metallic

- No significant thermal loading that changes preload is present during mechanical loading.

- The joint is not a tension joint in which gapping cannot be tolerated. A “tension joint” is defined as a joint in which the largest component of the applied load is tension.

- The fastener is in a local pattern of two or more fasteners.

- Fastener prying effects are correctly accounted for.

Torque (T) can be estimated with the following equation:
$$
T = KDP
$$
where: 
T = Torque (inch-lbs)
K = Torque Coefficient, also called Nut Factor: Factor representing the effect of friction and
geometry in the preload to torque relation.
D = Nominal Fastener Diameter (inches)
P = Preload (lbs)
The Preload “P” is calculated based on material yield strength “ “(typically 65% of yield)
times the stress area of the fastener “A”.
       
where: n = number of threads per inch
	 
The allowable tolerance range on the values is $\pm5%$ of the target torque value.


In [ ]:

#####  User input  ######
BoltMaterial = "A286"
Bolt = Mech.BoltSize("#4","Socket Cap Head")  # MMC of fastener; NAS1352
Bolt_material_spec = Mech.MaterialDefinition(BoltMaterial)
K = Mech.NutFactor('Lubricated')

DesiredLoad = 80
###########################

D_maj = Bolt["Diameter"]
n = Bolt["Pitch"]

ys = Bolt_material_spec["Yield strength"] # yield strength of fastener;
Uncertainty = K

AllowableTolerance = .05; # [%]

A = np.round(np.pi/4*(D_maj-.9743/n)**2,3)  # rounding done to match PG table values; tensile stress area [in^2]

# if bolt has a reduced shank diameter the Area use should be of the
# reduced shank, not the bolt's threaded area
# A = pi*(.1/2)^2  
if Bolt["Diameter"] < .112:
    Factor = .6
else:
    Factor = .65

# section used to match LGA patch toque; further research needed to fully assess why the increase factor.  Thermal(?) 
#if BoltMaterial ==  "AL6061":
#    Factor = Factor+.12

P = np.round(Factor*ys*A,1) # " 0.65 is 65% of yield strength of bolt material; Clamp force [lbf]
print ('Bolt clamp force (pre-load) %3.3f [lbf]' % P)

T = np.round(K*D_maj*P,1) # [lbf-in]
print('Torque required for estimated pre-load: %3.3f [lbf-in]' % T)

P_max = P*(1+AllowableTolerance)
T_max = np.round(K*D_maj*P_max,1)
P_min = P*(1-AllowableTolerance )
T_min = np.round(K*D_maj*P_min,1)
uts = Bolt_material_spec[1]
F_su = .75*uts; # [lb/in^2]
P_ult = F_su*A # [lbf]

print('Allowable pull-out strength: %6.2f [lbs]'% np.min(P_ult))

#### Force Estimates based on Torque

In [ ]:

#####  User input  ######
BoltMaterial = "AL6061"
Bolt = Mech.BoltSize("#10","Socket Cap Head")  # MMC of fastener; NAS1352
Bolt_material_spec = Mech.MaterialDefinition(BoltMaterial)
K = Mech.NutFactor['Arathane']

Torque = 17 # [lbf-in]

###########################

D_maj = Bolt["Diameter"]
n = Bolt["Pitch"]

ys = Bolt_material_spec["Yield strength"] # yield strength of fastener;
Uncertainty = .25

AllowableTolerance = .05; # [%]

A = np.round(np.pi/4*(D_maj-.9743/n)**2,3)  # rounding done to match PG table values; tensile stress area [in^2]

# if bolt has a reduced shank diameter the Area use should be of the
# reduced shank, not the bolt's threaded area
# A = pi*(.1/2)^2  
if Bolt["Diameter"] < .112:
    Factor = .6
else:
    Factor = .65

# section used to match LGA patch toque; further research needed to fully assess why the increase factor.  Thermal(?) 
#if BoltMaterial ==  "AL6061":
#    Factor = Factor+.12

P = T/(K*D_maj) # [lbs]

print('Estimated pre-load for given torque: %3.3f [lbs]' % P)




### Running torque

From: 540-PG-8072.1.2B: Mechanical Fastener Torque Guidelines

<img src="images/TorqueRangeLockingFeatures.jpg" height = 900>

If a fastener locking system does not meet these requirements, then the locking feature, nut, insert or bolt should be replaced in order to re-gain the prevailing torque required. If replacement is not practical, then it may be acceptable to apply a liquid locking compound per the requirements of section 1.5 below in order to comply with GSFC-STD-1000 Rule 4.20.

In [ ]:
#####  User input  ######
insertSize = '4-40'

Insert1 = 1.75  # [in-lbs]
Insert2 = 1.75  # [in-lbs]
Insert3 = 1.5  # [in-lbs]
Insert4 = 1.75  # [in-lbs]
Insert5 = 0  # [in-lbs]
Insert6 = 0  # [in-lbs]
Insert7 = 0  # [in-lbs]
Insert8 = 0  # [in-lbs]

###########################

match insertSize:
   case '2-56':
      Locking_insert_limits = [3/16,2]
   case '4-40':
      Locking_insert_limits = [10/16,3]
   case '6-32':
      Locking_insert_limits = [1,6]
   case '8-32':
      Locking_insert_limits = [1.5,9]
   case '10-24':
      Locking_insert_limits = [2,13]
      

Inserts = np.array([Insert1,Insert2,Insert3,Insert4,Insert5,Insert6,Insert7,Insert8])

Idx = np.zeros(Inserts.size)
cnt = 0
flag = 0
for i in range(Inserts.size):
   if Inserts[i] != 0:
      cnt = cnt + 1
      if (Inserts[i] > Locking_insert_limits[1]) or (Inserts[i] < Locking_insert_limits[0]):
         flag = flag + 1


Avg_runningTorque = np.sum(Inserts)/(cnt)

if flag > 0:
   print ('Running torque out of spec recorded!!')
else:
   print('Average runnnig torque is: %2.3f [in-lbs]'%Avg_runningTorque)

### Bolt Size estimate

Here's a rough guide for picking a screw or bolt for a given load:

Start off with the load that needs to be held in tension, call this F. If you have a shear (sideways) load, you should design so that friction or dowell pins will bear the load and not the bolt, but if this isn't an option note that shear strength is 60% of tensile strength in many steels.

We'll use a safety factor of 2.5, so the design load is now 2.5F. Now we need to select a screw with enough strength so that it can withstand the combined external load and pre-load from tightening. Assuming that 80% of the bolt's proof strength is being used up in preload, that leaves 20% to handle 1/3 of the external load. Or in other words, we're looking for a bolt where 60% of its proof strength is greater than the load.

Let's try an example: What size grade 2 bolt is necessary to hold 100 lbs? The proof strength of Grade 2 bolts between .24 and .75 inches is 55 ksi (thousand pounds per square inch), and 60% of this is 34.2 ksi. So, we're looking for a bolt with a tensile area greater than our load (2.5*100 lbs) divided by 33 ksi, or .0076 square inches. A #6 UNC should work. For perspective the diameter of a #6 screw is .138", (1/8 = .125"). If this seems small, keep in mind that the ultimate strength (breaking strength) of a Grade 2 bolt is 74 ksi, so a #6 screw could theoretically hold 672 lbs in pure tension. If you're wondering why bolts you see in cars and weight machines are so large, it's partly to gaurd against loosening and fatigue failure in addition to safety factors.

<img src="images/ScrewSizeTable.jpg" height = 400>

In [ ]:

Load = 100 # [lbs]

bolt_strength = 55 # [ksi]
proof_strenght_factor = 0.6; # 60[%]
safety_factor = 2.5 #

bs_sf = bolt_strength*proof_strenght_factor*1e3 # [psi]
Load_n = Load*safety_factor

Needed_area = Load_n/(bs_sf) # [square inches]

# A286 SS = 160ksi
bolt_strength = 160 # [ksi]
bolt_d = 0.138  # bolt diameter [in]

bold_load_capacity = proof_strenght_factor*(bolt_strength*1e3)*(np.pi*(bolt_d/2)**2) # [Lbs]
# bold_load_capacity = 0.6/.4*(bolt_strength*1e3)*(pi*(bold_d/2)^2)


print(bold_load_capacity)

## Preloaded Joint Analysis Methodology for Space Flight Systems

This source of this page is Chambers, Jeffrey A., "Preloaded Joint Analysis Methodology for Space Flight Systems," NASA Technical Memorandum 106943, 1995.

### The Preloaded Joint

#### Determining Bolt Preload
In general, a bolted joint performs best when it is preloaded such that the working loads are reacted primarily by the portion of the joint in compression. If designed properly, the bolt actually carries only a small portion (usually less than 20 percent) of that external loading while the greater portion of loading is offset by the release of the compressive energy introduced to the flanges during torquing. Essentially, a large portion of work is performed by the joint while a small portion of work is performed on the bolt. The joint is initially placed in compression by applying a tensile preload to the bolt. An initial preload is introduced so that the compression in the flanges is never completely relieved and hence the flange faces never separate. In order to obtain this level of preload, the bolt is usually prestressed very near its working limits (usually 65 to 90 percent of its yield strength). 

$$
P_0 = \frac{T}{KD}(1.0\pm u)
$$

The basic preload may vary from the intended value, either more or less, by an amount established by the preload uncertainty factor, $u$. In general, it is safe to assume that the preload uncertainty for a hand-operated torque wrench used on a lubricated fastener is ±25 percent.

The applied torque, $T$, and nominal diameter, $D$, are generally known and measurable parameters, but the nut factor, $K$, is not. The nut factor is essentially a factor applied to account for the effects of friction in the torquing elements (both in the threads and under the bolt head/nut).   Unfortunately, calculating $K$ is quite complex since frictional coefficients between heavily loaded parts are not easily estimated with accuracy. A simpler approach is to assume that the nut factor usually ranges from 0.11 to 0.15 for lubricated fasteners. **GSFC uses 0.18**

For unlubricated fasteners, a nut factor on the order of 0.2 may be used. When selecting a nut factor, the engineer may wish to examine both extremes of a reasonable range in order to assess the impacts on joint design. **GSFC uses 0.3**

As an alternative to the typical nut factor method of determining preload, the torque-preload relationships can be determined experimentally. Here, the torque-preload relationships are determined by direct measurements taken from instrumented joint specimens. Statistical data is recorded for the torque required to achieve a desired bolt force. Many relationships have been developed for various sizes, types, lubrications, and bolt materials commonly used in space flight hardware and are well documented in MSFC-STD-486B. For tensile loading applications, if the fastener is torqued in accordance with the guidelines, it may be assumed that the pre-tensioning develops *65 percent of the tensile yield strength of the bolt material*.

$$
\frac{T}{KD} = 0.65 F_{ty}A_t
$$

$A_t$ is the fastener tensile cross section and $F_{ty}$ is the fastener material yeild strenght.

$$
A_t = \frac{\pi}{4} (D-0.9743/n)^2
$$

where $n$ is the number of threads per inch.  

Torquing and preload uncertainty, however, are not the only parameters affecting the initial joint preload. Temperature changes and preload relaxation can modify initial preload. Thermal loading on the joint may be experienced if the bolt and flange materials have different coefficients of thermal expansion and the joint is subjected to a temperature change. Under a given temperature change (measured from the assembly state) the bolt and abutments expand or contract at differing rates which introduces a tension or relaxation in the bolt ($P_{th}$). Small changes in global operating conditions or large local temperature gradients can result in significant changes in joint loading and therefore must be considered.

To maintain contact within the joint, the thermal deflection of the bolt must be balanced by the total deflection in the flanges. For purposes of developing the relation, assume a connection with flanges made of aluminum and a bolt made of steel ($\alpha_j > \alpha_b$) is subjected to a uniform temperature increase. The flanges attempt to expand more than the bolt will allow which increases the load in the bolt. Therefore the total elongation of the bolt is the result of two components: the unrestricted thermal elongation plus an elongation due to the increased load in the joint. The total deflection in the flanges is the difference between the unrestricted thermal expansion and the additional compression due to the increase in preload. The change in preload can be derived as shown below.

$$
\delta_b = \frac{P_{th}}{K_b} + \alpha_b L \Delta Temp
$$

$$
\delta_j = \frac{-P_{th}}{K_j} + \alpha_j L \Delta Temp
$$

Since the bolt and joint deflections are equal, $\delta_b = \delta_j$

$$
P_{th} = \frac{K_b K_j}{K_b + K_j} L \Delta Temp (\alpha_j - \alpha_b)
$$

Figure below illustrates how a temperature change affects the initial joint load.

<img src="images/Figure2.jpg" height = 300>

Preload relaxation (or embedding) occurs as the contact surfaces of the flanges and joining elements experience local yielding as they conform to one another over a period of time. There may also be some localized yielding in the threads of the bolt and nut that results in additional relaxation. . Preload relaxation can also be encountered if elastomeric joint materials (e.g., gaskets) are used and experience permanent set over time. Dynamic or cyclic loading can lead to settling in the joint through fretting of the contact surfaces. The amount of preload relaxation can be quite difficult to characterize since it must consider the materials, loading, and physical (e.g., corrosive) environment in which the joint exists. The amount of embedding for typical metal-to-metal joints in a noncorrosive environment is typically between 2 and 10 percent. For design and analysis purposes it is safe to assume the preload loss to be about 5 percent, that is,

$$
P_{relax} = 0.05 P_{0,min}
$$

<img src="images/Figure3.jpg" height = 300>

### Fastener Axial Load

The total axial load in a fastener consists of the preload plus that portion of the external mechanical load not reacted by the joint. The total axial bolt load, Pb, can be given 

$$
P_b = P_{0,max} + (SF \times n \phi P_{et})
$$

where $P_{et}$ is the resultant external force directed at the joint. This can be obtained through a free-body diagram of the system, finite element results, or other means. This external force must however include all components (e.g., prying action, moment resistance, etc.) that may increase or decrease the final force acting at the bolt. A factor of safety (SF) is applied to the external loading only (as opposed to $P_b$ as a whole) since inaccuracies of the preloading process have already been accounted for in the development of Po. The factors of safety for general space flight hardware are usually dependent on the method of verification used (ref. 5) and may differ from program to program. For nonpressurized, untested applications the safety factors are normally 1.25 and 2.0 for yield and ultimate strengths, respectively, while 1.1 and 1.4 are typical for nonpressurized, tested applications. Safety factors are strongly dependent on the specific application, method of loading, and overall design requirements, and therefore should be reviewed carefully before using them with the joint equations.

<img src="images/Figure4.jpg" height = 300>

For common joint designs the load is carried somewhere near the midplanes of the flanges as shown in fig. 6. With loading introduced near these midplanes (n = 0.5), the flange regions inboard and outboard of the loading planes work together much like the case of n $\approx$  0 but to a lesser degree. The loading plane factor is described as

$$
n = \frac{distance \ between \ planes}{total \ thickness \ of \ joint}
$$

For most joints, it is usually acceptable to assume the loading planes to be located at the midplanes of the flanges or the midplanes outermost members if more than two components are being bolted. The joint configuration should always be examined closely to insure that this assumption is applicable.

The stiffness factor, $\phi$, determines the proportion in which the load is shared between bolt and joint. Since the extensional deflection, $\delta$, of the bolt under an arbitrary tensile loading, Pet, is equal to the amount of net deflection in the flanges, the force in each component can be determined with the aid of the load-deflection diagram (fig. 6).

<img src="images/Figure6.jpg" height = 400>

$$
P_{et} = \Delta P_b + \Delta P_j = \Delta P_b \left(\frac{K_b + K_j}{n K_b} \right)
$$

So the stiffness factor (or load factor) is defined as

$$
\phi = \frac{K_b}{K_b + K_j}
$$

The bolt stiffness, $K_b$, is equal to the axial stiffness of a circular rod with a cross section based on the nominal bolt diameter. The joint stiffness, $K_j$, is taken as the stiffness of the flange region which experiences the compressive preload. It can be very difficult to determine the exact region of the flange which is placed in compression and equally difficult to determine its stiffness. Several methods exist that estimate (either mathematically or experimentally) the stiffness of this load affected region; however, the method outlined by *Shigley* has been used in this report. This method assumes that compressive loading in the flange(s) is distributed through 45° conical sections like those shown in fig. 6. Relations for the various joint parameters are given for several typical joint configurations shown in figures 7 to 10.

In [47]:

#####  User input  ######
FastenerInstall = 'Dry'  # 'Dry', 'Lubricated', 'Arathane'
BoltNumber = "#2"
BoltType = 'Socket Cap Head' #  ; 'Flat Head 100 deg'
Bolt = Mech.BoltSize(BoltNumber,BoltType)  # MMC of fastener; NAS1352
Insert = Mech.InsertSize(BoltNumber)
Washer_OD = .096  #[in]

Bolt_material_spec = Mech.MaterialDefinition("CRES")
Joint_material_spec = Mech.MaterialDefinition("RO4003C")
Joint_material_spec2 = Mech.MaterialDefinition("SSP2368-65")

InsertParent_material_spec = Mech.MaterialDefinition("AL6061")
K = Mech.NutFactor[FastenerInstall]
#K = NutFactor('Arathane')
Fastener_Engagement = 1.5;  # fastener engagement in diameters
ClampMaterialThickness1 = 0.096 # Clamped Material Thickness [in]
ClampMaterialThickness2 = 0.050; # Clamped Material Thickness [in]

Insert_material_spec = Mech.MaterialDefinition("CRES")
InsertLength = 1.5*Bolt["Diameter"]

ParentMaterialThickness = InsertLength; # Insert parent Material Thickness [in]; should be > insert length

FactorBolt = .65  # .75 for metals, .5 for polymers
FactorInsert = .65  # .75 for metals, .5 for polymers
FactorParentMaterial = .65  # .75 for metals, .5 for polymers

TargetBoltForce = 81 # [lbs] ; use 0 if unknown

N_material = 2

match N_material:
    case 1:
        # single material
        layersThickness = np.array([ClampMaterialThickness1,ParentMaterialThickness])
        layersMaterialModulusElasticity = np.array([Joint_material_spec['Young modulus'],InsertParent_material_spec['Young modulus']])
        layersMaterialThermalConductivity = np.array([Joint_material_spec['Linear CTE']])
        layersMaterialTensileStrength = np.array([Joint_material_spec['Ultimate tensile strength']])
        layersMaterialYieldStrength = np.array([Joint_material_spec["Yield strength"]])
        layersType = [Joint_material_spec['Type']]
    case 2:
        # dual material
        layersThickness = np.array([ClampMaterialThickness1,ClampMaterialThickness2,ParentMaterialThickness])
        layersMaterialModulusElasticity = np.array([Joint_material_spec['Young modulus'],Joint_material_spec2['Young modulus'],InsertParent_material_spec['Young modulus']])
        layersMaterialThermalConductivity = np.array([Joint_material_spec['Linear CTE'], Joint_material_spec2['Linear CTE']])
        layersMaterialTensileStrength = np.array([Joint_material_spec['Ultimate tensile strength'],Joint_material_spec['Ultimate tensile strength']])
        layersMaterialYieldStrength = np.array([Joint_material_spec['Yield strength'],Joint_material_spec2['Yield strength']])
        layersType = [Joint_material_spec['Type'],Joint_material_spec2['Type']]

SafetyFactor = 1.25  # 2 

if FastenerInstall == "Dry":
    Uncertainty = .35
else:
    Uncertainty = .25

Configuration = 3

T0 = 20  # [C]
T_end = -25  # [C]
DeltaTemp = units.C2F(T_end) - units.C2F(T0)  # [F]

NumberOfBolts = 7  # number of bolts in the assembly
ClampArea = float(2.26) # [in^2]  # area of the clamped material; used to calculate the pressure applied by the fastener
###########################

#### Configuration 1

Multiple parts are bolted together with a through-bolt and washer/nut combination. The bolt may be hex, socket, or pan head style (see fig. 7).

<img src="images/Figure7.jpg" width = 600>

#### Configuration 2

Parts are bolted together with a flat-head through-bolt and washer/nut combination (see fig. 8).

<img src="images/Figure8.jpg" width = 600>

Note that, if $d_w \approx 1.5D$, which is the case for typical aerospace fasteners, then equation (38) reduces to equation (33).

#### Configuration 3

Parts are bolted together with a bolt threaded into the last part (with or without insert). The bolt may be hex, socket, or pan head style (see fig. 9).

<img src="images/Figure9.jpg" width = 600>

#### Configuration 4

Parts are bolted together using a flat-head bolt threaded into the last assembled part (see fig. 10):

<img src="images/Figure10.jpg" width = 600>


In [44]:
match Configuration:
    case 1:
        if BoltType == "Flat Head 100 deg":
            print('Bolt type cannot be flat head 100 deg')
        else:
            print("equations need to be verified!!")
            # eq 31
            L = np.sum(layersThickness)
            nx = layersThickness.size

            # eq 32
            D_maj =Bolt['Diameter']
            Ab = np.round(np.pi/4*(D_maj-.9743*(1/Bolt['Pitch']))**2,3) # [in^2]
            Eb = Bolt_material_spec['Young modulus'] # [psi] Modulus of elasticity
            Lb = L # bolt length [in]
            kb = Ab*Eb/Lb  # bolt stiffness [in * ksi] = [in * lbf/in^2] = [lbf/in] = Force/length

            # eq 34
            L_E_ratio = np.zeros(layersThickness.size,dtype=float, order='F')
            
            for i in range(nx):
                L_E_ratio[i] = layersThickness[i]/layersMaterialModulusElasticity[i]

            Ej = L/np.sum(L_E_ratio)

            # eq 33
            kj = (np.pi*Ej*D_maj)/(2*np.log(5*(L+0.5*D_maj)/(L+2.5*D_maj)))


            if nx==2:
                n = (layersThickness[0]/2 + (layersThickness[-1]/2))/np.sum(layersThickness)
            else:
                n = (layersThickness[0]/2 + np.sum((layersThickness[1:nx-1])) + (layersThickness[-1]/2))/np.sum(layersThickness)

    case 2:
        if BoltType == "Flat Head 100 deg":
            # eq 36
            print("equations need to be verified!!")
            nx = layersThickness.size
            L = np.sum(layersThickness[0:nx-1]) + (layersThickness[nx] - Bolt['Head Height']/2)
            

            # eq 37
            D_maj =Bolt['Diameter']
            Ab = np.round(np.pi/4*(D_maj-.9743*(1/Bolt['Pitch']))**2,3)  # [in^2]
            Eb = Bolt_material_spec['Young modulus'] # [psi] Modulus of elasticity
            Lb = L # bolt length [in]
            kb = Ab*Eb/Lb  # bolt stiffness [in * ksi] = [in * lbf/in^2] = [lbf/in] = Force/length

            # eq 39
            dw = (Bolt['Head Diameter'] + D_maj)/2 

            # eq 40
            L_E_ratio = np.zeros(layersThickness.size,dtype=float, order='F')
            for i in range(nx):
                if i==0:
                    L_E_ratio[i] = (layersThickness[i] - Bolt['Head Height']/2) / layersMaterialModulusElasticity[i]
                else:
                    L_E_ratio[i] = layersThickness[i] / layersMaterialModulusElasticity[i]

            Ej = L/np.sum(L_E_ratio)

            # eq 38
            kj = (np.pi*Ej*D_maj)/np.log(((L+dw-D_maj)*(dw+D_maj)*(L+0.5*D_maj))/((L+dw+D_maj)*(dw-D_maj)*(L+0.5*D_maj)))
            
            # eq 41
            n = ((layersThickness[0] - Bolt['Head Height']/2) + np.sum((layersThickness[1:nx-1])) + layersThickness[nx]/2)/np.sum(layersThickness)
        else:
            print('Bolt type must be flat head 100 deg')
            
    case 3:
        if BoltType == "Flat Head 100 deg":
            print('Bolt type cannot be flat head 100 deg')
        else:
            nx = layersThickness.size

            # eq 42
            Li = Fastener_Engagement*Bolt['Diameter'] # insert length (thread engagement)

            L = np.sum(layersThickness[0:nx-1]) + (layersThickness[-1] - Li/2)
        
            # eq 43
            D_maj =Bolt['Diameter']
            Ab = .7854*(D_maj-.9743/Bolt['Pitch'])**2
            Eb = Bolt_material_spec['Young modulus'] # bolt modulus of elasticity, [psi]
            Lb = L# bolt length [in]
            kb = Ab*Eb/Lb  # bolt stiffness [in * psi] = [in * lbf/in^2] = [lbf/in] = Force/length
            Kb = kb*NumberOfBolts  # total bolt stiffness [in * psi] = [in * lbf/in^2] = [lbf/in] = Force/length
            
            # eq 45
            L_E_ratio = np.zeros(layersThickness.size,dtype=float, order='F')
            for i in range(nx):
                if i==nx-1:
                    L_E_ratio[i] = (layersThickness[i]-Li/2)/layersMaterialModulusElasticity[i]
                else:
                    L_E_ratio[i] = layersThickness[i]/layersMaterialModulusElasticity[i]

            Ej = L/np.sum(L_E_ratio)

            # eq 44
            kj = (np.pi*Ej*D_maj)/(np.log(5*(2*L+0.5*D_maj)/(2*L+2.5*D_maj)))

            # this equations match Peter Mule estimates...  need to look why the difference...
            
            k_eqj = 1/((layersThickness[0]/(layersMaterialModulusElasticity[0]*ClampArea)) + (layersThickness[1]/(layersMaterialModulusElasticity[1]*ClampArea)))
            # eq 46
            if nx==2:
                n = (layersThickness[0]/2 + (layersThickness[-1]-Li/2))/np.sum(layersThickness)
            else:
                n = (layersThickness[0]/2 + np.sum((layersThickness[1:nx-1])) + (layersThickness[-1]-Li/2))/np.sum(layersThickness)

            # thermal effects added
            alpha_b = Bolt_material_spec['Linear CTE']
            alpha_j = np.zeros(layersMaterialThermalConductivity.size,dtype=float, order='F')
            #for i in range(nx-1):
            ### This should be a for loop for all joint materials
            alpha_j_eq = np.sum(layersMaterialThermalConductivity*layersThickness[0:nx-1])/np.sum(layersThickness[0:nx-1])
            #alpha_j_eq = 
            P_thermal = (kb*kj)/(kb+kj)*L* DeltaTemp * (alpha_j_eq - alpha_b) # preload loss
            P_thermal_pm = (kb*7*k_eqj)/(kb*7+k_eqj)*L* DeltaTemp * (alpha_j_eq - alpha_b)

    case 4:
        if BoltType == "Flat Head 100 deg":
            print("equations need to be verified!!")
            # eq 47
            Li = Fastener_Engagement*Bolt['Diameter'] # insert length (thread engagement)
            nx = layersThickness.size
            L = (layersThickness[0] - Bolt['head Diameter']/2) + np.sum(layersThickness[1:nx-1]) + (layersThickness[nx] - Li/2)
            
            # eq 48
            D_maj =Bolt['Diameter']
            Ab = np.round(np.pi/4*(D_maj-.9743*(1/Bolt['Pitch']))**2,3)  # [in^2]  # [in^2]
            Eb = Bolt_material_spec['Young modulus'] # [psi] Young modulus
            Lb = L # bolt length [in]
            kb = Ab*Eb/Lb  # bolt stiffness [in * ksi] = [in * lbf/in^2] = [lbf/in] = Force/length

            # eq 50
            dw = (Bolt['Head Diameter'] + D_maj)/2 

            # eq 51
            L_E_ratio = np.zeros(layersThickness.size,dtype=float, order='F')
            for i in range(nx):
                if i==0:
                    L_E_ratio[i] = (layersThickness[i] - Bolt[3]/2) / layersMaterialModulusElasticity[i]
                elif i==nx:
                    L_E_ratio[i] = (layersThickness[i] - Li/2) / layersMaterialModulusElasticity[i]
                else:
                    L_E_ratio[i] = layersThickness[i] / layersMaterialModulusElasticity[i]

            Ej = L/np.sum(L_E_ratio)

            # eq 49
            kj = (np.pi*Ej*D_maj)/np.log(((L+dw-D_maj)*(dw+D_maj))/((L+dw+D_maj)*(dw-D_maj)))
            
            # eq 52
            n = ((layersThickness[0] - Bolt['Head Height']/2) + np.sum((layersThickness[1:nx-1])) + (layersThickness[nx] -Li/2))/np.sum(layersThickness)
        else:
            print('Bolt type must be flat head 100 deg')

AllowableTolerance = .0; # [%]

A = np.round(np.pi/4*(D_maj-.9743*(1/Bolt['Pitch']))**2,3)  # rounding done to match PG table values; tensile stress area [in^2]

# if bolt has a reduced shank diameter the Area use should be of the
# reduced shank, not the bolt's threaded area
# A = pi*(.1/2)^2  
if Bolt["Diameter"] < .112:
    Factor = .6
else:
    Factor = .65

# section used to match LGA patch toque; further research needed to fully assess why the increase factor.  Thermal(?) 
#if BoltMaterial ==  "AL6061":
#    Factor = Factor+.12

P = np.round(Factor*Bolt_material_spec["Yield strength"]*A,1) # " 0.65 is 65% of yield strength of bolt material; Clamp force [lbf]
if TargetBoltForce>0:
    P = TargetBoltForce
else:
    print ('Bolt clamp force (pre-load) %3.3f [lbf]' % P)

T = np.round(K*D_maj*P,1) # [lbf-in]
print('Torque required for estimated pre-load: %3.3f [lbf-in]' % T)

# P_relax = 0.05*P_min

Po_max = P*(1+AllowableTolerance) + np.sum(P_thermal)

boltDeflection = Po_max/kb
jointDeflection = Po_max/kj

db = P/kb  # thermal deflection of the bolt [in]
dj = P/kj  # thermal deflection of the joint [in]
Delta_Pb = P_thermal
delta = Delta_Pb *((db+(1+n)*dj)/(P))
Delta_Pj = delta*P/(n*dj)
P_external = Delta_Pb + Delta_Pj  # total external load [lbs]

phi = kb/(kb+kj) # stiffness factor

P_max = P*(1+Uncertainty)
P_min = P*(1-Uncertainty)

#P_b = P_min + (SafetyFactor*n*phi*P_external)
P_b = P_min + (P_external)

ExpectedDeflection = P_b/kj

separationSafetyFactor = 1.25
if Delta_Pj > 0:
    MS = P/(separationSafetyFactor*np.abs(Delta_Pj)) - 1
print("Estimated nominal force %3.3f [lbs]" % P)

print("Estimated added force due to thermal effects %3.3f [lbs]" %  np.sum(P_external))

#print("Estimated final force on the bolt %3.3f [lbs]" % P_final)

print("Estimated total axial load in fastener %3.3f [lbs]" % P_b)
print("Estimated clamped material deflection: %3.6f [in] or %3.3f [percent]" % (ExpectedDeflection,ExpectedDeflection/np.sum(layersThickness)*100))

C = kb/(kb+kj)
Fb_load_percent = C
Fm_load_percent = 1-C
separationFactor = 1/(1-C)  # needs to be >1

A = np.max([(np.pi*(Washer_OD/2)**2),(np.pi*(Bolt["Head Diameter"]/2)**2)]) - np.pi*(Bolt["Diameter"]/2)**2

Stress = P_b * A  # [psi]
if "Dielectric" in layersType:
    if Stress > np.min(0.55*layersMaterialTensileStrength):
        print("Caution!")
        print("Stress exceeds allowable values")
    
    print("Estimated Stress is %3.3f times ultimate tensile strength." % (Stress/np.min(layersMaterialTensileStrength)))
else:
    if Stress > np.min(0.65*layersMaterialYieldStrength):
        print("Caution!")
        print("Stress exceeds allowable values")

    print("Estimated Stress is %3.3f times ultimate yield strength." % (Stress/np.min(layersMaterialYieldStrength)))

Torque required for estimated pre-load: 2.100 [lbf-in]
Estimated nominal force 81.000 [lbs]
Estimated added force due to thermal effects -3.015 [lbs]
Estimated total axial load in fastener 57.735 [lbs]
Estimated clamped material deflection: 0.048891 [in] or 17.779 [percent]
Estimated Stress is 0.000 times ultimate tensile strength.


### Threaded Insert Analysis

For joints using threaded inserts, such as the joint shown in fig. 14, three basic modes of failures may be encountered. The first mode of failure, shear failure of the insert's internal threads, is exhibited as the fastener pulls out of the insert, failing the internal threads of the insert. The second failure mode, shear failure of the insert's external threads, is exhibited as the insert pulls from the parent material, failing the external threads of the insert. The third mode of failure, shear failure of the parent material's internal threads, results as the fastener and insert together pull from the parent material, failing the internal threads of the parent material. Each failure mode may be investigated using the methods described in the following sections.

<img src="images/Figure14.jpg" width = 400>


#### Insert Internal Thread Failure

The ultimate strength of the insert in the internal thread shear failure mode is dependent on the amount of shear area available to resist axial loading of the bolt. This thread shear area is a function of the thread size and type as well as the length of thread engagement. In much the same manner as the external thread shear strength of the bolt, the insert internal thread shear strength is based on the major diameter of the mating external threads. This thread shear area can be estimated by (ref. 8).
$$

A_s = \frac{3\pi L_e D_{major,ext}}{4} \ (76)

$$

The insert ultimate allowable pull-out strength is then
$$
P_{ult} = F_{su} A_s 
$$

$F_{su}$ is the ultimate shear strength (typically .75*ultimate tensile strength) for SS and AL. about .5 for PEEK and PTFE.

**For this document polymers will have the 50% ratio and metals will use 75%.  Ths needs to be revised later...**

#### Insert External Thread Failure

The shear area of the insert's externally threaded region is calculated in the same manner as that for the external thread shear area of the bolts given by equation (63) with the exception that the area must be reduced by the amount of area lost for the insert locking keys (if applicable). With external thread shear area, the insert pull-out strength is
$$
A_s = \frac{5\pi L_e D_{minor,int}}{8}  \ (63)
$$
$$
P_{ult} = F_{su}A_s  \ (78)
$$

#### Insert Parent Material Thread Failure
Although the actual thread shear area of the parent material is increased slightly over that of the insert's external thread shear area, for conservative purposes the shear area of the parent material internal thread is assumed to be the same as the insert's reduced external thread shear area used for equation (78). The parent material pull-out strength is then

$$
A_s = \frac{5\pi L_e D_{minor,int}}{8}

P_{ult} = F_{su}A_s

$$
          

#### Margin of Safety Criteria

For all three modes of failure, the margin of safety is given as

$$
MS = \frac{P_{ult}}{P_b} - 1
$$

The margin of safety should be calculated for all three modes of failure, for ultimate strength only, to determine the limiting mode of failure.


In [ ]:
#####  User input  ######

NewInfo = 0

if NewInfo == 1:
    BoltMaterial = "A286"
    BoltNumber = "#2"

    Bolt = Mech.BoltSize(BoltNumber,"Socket Cap Head")  # MMC of fastener; NAS1352
    Bolt_material_spec = Mech.MaterialDefinition(BoltMaterial)
    K = Mech.NutFactor['Lubricated']

    Parent_material_spec = Mech.MaterialDefinition("AL6061")

    Insert = Mech.InsertSize(BoltNumber)
    Insert_material_spec = Mech.MaterialDefinition("A286")
    InsertLength = 1.5*Bolt["Diameter"]

    Fastener_Engagement = 1.5;  # fastener engagement in diameters

    # use information from previous analysis

###########################
P_ult = np.zeros(3)

D_maj = Bolt["Diameter"]
n = Bolt["Pitch"]

ys = Bolt_material_spec["Yield strength"] # yield strength of fastener;
Uncertainty = 0.25


AllowableTolerance = .05; # [%]

A = np.round(np.pi/4*(D_maj-.9743/n)**2,3)  # rounding done to match PG table values; tensile stress area [in^2]

# if bolt has a reduced shank diameter the Area use should be of the
# reduced shank, not the bolt's threaded area
# A = pi*(.1/2)^2  

if Bolt["Diameter"] < .112:
    Factor = .6
else:
    Factor = .65

# section used to match LGA patch toque; further research needed to fully assess why the increase factor.  Thermal(?) 
#if BoltMaterial ==  "AL6061":
#    Factor = Factor+.12

P = np.round(Factor*ys*A,1) # " 0.65 is 65% of yield strength of bolt material; Clamp force [lbf]
print ('Bolt clamp force (pre-load) %3.3f [lbf]' % P)

T = np.round(K*D_maj*P,1) # [lbf-in]
print('Torque required for estimated pre-load: %3.3f [lbf-in]' % T)

P_max = P*(1+AllowableTolerance)
T_max = np.round(K*D_maj*P_max,1)
P_min = P*(1-AllowableTolerance )
T_min = np.round(K*D_maj*P_min,1)
uts = Bolt_material_spec["Ultimate tensile strength"]
F_su = .75*uts; # [lb/in^2]
P_ult0 = F_su*A # [lbf]
print(P_ult0)

###  Uses input and P_b used in previous section ####
P_b = P

###
D_maj_f = Bolt["Diameter"]

Pitch = 1/n

H = np.sqrt(3)/2 * Pitch
D_p = D_maj_f - 3*np.sqrt(3)/8 * Pitch
D_min = D_maj_f - (5*np.sqrt(3)/8) * Pitch
Le = Fastener_Engagement*D_maj_f

Fsu = Factor*Bolt_material_spec['Ultimate tensile strength'];  # ultimate strength used for shear
As = (5*np.pi*Le*D_min)/8
# As = round(.7854*(D_maj_f-.9743/n)^2,3)  % rounding done to match PG table values
P_ult[0] = Fsu*As

D_maj_i = Insert["MinDiameter"]
Fsu = Factor*Insert_material_spec['Ultimate tensile strength']
As = (3*np.pi*Le*D_maj_i)/4

P_ult[1] = Fsu*As

Fsu = Factor*Parent_material_spec["Ultimate tensile strength"]
As = (5*np.pi*Le*D_min)/8
P_ult[2] = Fsu*As; # [lbs]
print(P_ult)
print('Allowable pull-out strength: %6.2f [lbs]',np.min(P_ult))

T = np.round(K*D_maj_f*np.min(P_ult)*.65,1)

MS = np.max(P_ult)/P_b - 1

print('Margin of safety for fastener is: %f' % MS)

### Fastener Strength Criteria and Modes of Failure

In general, for preloaded joints to work effectively they must meet (at a minimum) the following criteria (ref. 1):
1.	Bolt(s) and joint must have adequate strength.
2.	Joint must not experience separation under loading.
3.	Bolt(s) must have adequate fracture and fatigue life

In most applications the bolted connections in space flight hardware are considered to be statically loaded. The dynamic load components present during the launch, orbit, and landing phases are usually short in duration and therefore replaced by equivalent static loads that would be developed by the dynamic events.

The first requirement is explicitly defined by the payload safety verification requirements associated with space flight hardware which mandates that all safety and fracture critical fasteners possess positive (>0.0) margins of safety for all modes of failure. These margins of safety (MS) for bolts under various states of loading can be expressed (but are not limited to) as follows:

#### Tension Only Criteria

For bolts subjected to pure tensile loading, the following is applicable:

$$
MS = \frac{Tensile Allowable}{P_b} - 1
$$

In general, for most modes of failure a margin of safety can be calculated for both ultimate and yield strengths. Both of these margins should be checked to determine which is limiting (critical) since a positive margin may exist for one while a negative margin exists for the other.


#### Shear Only Criteria
For bolts subjected to pure shear loading, the following is applicable:
$$
MS = \frac{Shear Allowable}{SF \times V } - 1
$$

The externally applied shear load, V, is again found by resolving all external shear loads into a resultant load acting at the individual fastener. The shear load usually has components determined from translational forces as well as components resulting from resisting moments in the joint. The allowable shear load can be given by

$$
Shear Allowable = F_{su}A_s
$$

where $F_{su}$ is the ultimate shear strength of the bolt. The shear yield strength, $F_{su}$, may be assumed to be $0.577F_{ty}. 

The shear area, As, is normally equal to the minimum tensile area, for example, As = At, unless the joint is designed such that the shear plane acts on the unthreaded shank of the fastener. If the shear plane acts solely through the unthreaded portion of the bolt, the shear area may be based on the nominal diameter.


#### Shear Tear out
Shear tear out is possible when the bolt is positioned near the free edge of one or more of the abutment components and is loaded in shear. The bolt fails the abutment by shearing (or tearing) the material between the hole and the free edge of the abutment. This type of failure is common with lug type fittings and thin sheet abutments. The ultimate shear out load is

$$
P_{ult} = F_{su}A_{shear}
$$

The available shear area is:

$$
A_{shear} = 2*t \left(e - \frac{D}{2} \right)
$$

where $t$ is the thickness of the sheet or lug, $e$ is the perpendicular distance from the hole centerline to the free edge of the sheet, and $D$ is the nominal fastener diameter (as shown in fig. 13). A factor of two is used in the calculation of the shear area since the tear out occurs along two planes; one on each side of the bolt. This area is quite conservative since it considers the shear planes acting along the shortest distance between the edge of the hole and edge of the sheet (across section a-a). 

<img src="images/Figure13.jpg" height = 300>

The associated margin of safety is then:

$$
MS = \frac{P_{ult}}{SF \times V} - 1
$$

The possibility of encountering shear tearout can be greatly reduced if design practices are employed which maintain minimum e/D ratios of 2.0 or more. Occasionally the hardware design does not permit maintaining the 2.0 factor and the ratio must be reduced. In this situation, the e/D ratio may be reduced to as low as 1.5, however, *it is never advisable to permit edge conditions resulting in an e/D ratio of less than 1.5* 


In [ ]:
V = 2 #[lbf]
P_added = 4#[lbf]

#TensionSafetyFactor = 2
#ShearSafetyFactor = 2.6

TensionSafetyFactor = 1.4
ShearSafetyFactor = 1.25

As = (5*np.pi*Le*D_min)/8 
Fty = Bolt_material_spec['Yield strength'] # material tensile yield strength [psi]
if Bolt_material_spec[3] == 0:
    Fsu = .75*Bolt_material_spec["Ultimate tensile strength"] # [psi]
else:
    Fsu = Bolt_material_spec["Ultimate shear strength"] # material shear strength [psi]

Fsu0 = 0.577*Bolt_material_spec["Yield strength"];  # ultimate strength used for shear

TensileAllowable = Fty*As
ShearAllowable = Fsu*As
print(TensileAllowable)

MS_tension = TensileAllowable/(P_b+P_added) -1
print('Margin of safety for fastener in tension is: %f' % MS_tension)

MS_shear = ShearAllowable/(ShearSafetyFactor*V) - 1
print('Margin of safety for fastener in shear is: %f' % MS_shear)

Rt = P_b/TensileAllowable
Rs = ShearSafetyFactor*V/ShearAllowable

MS_total = 1/np.sqrt(Rt**2 + Rs**3) - 1
print('Combined Margin of safety is: %f' % MS_total)


### Joint separation
